In [1]:
!pip install sentence-transformers chromadb groq pandas -q
print("Installation Completed")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 60.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 76.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 72.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 5.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not current

In [2]:
import pandas as pd
import chromadb
from sentence_transformers import SentenceTransformer
from groq import Groq
import os
print("All libraries imported successfully")

All libraries imported successfully


In [3]:
from chromadb import api
GROQ_API_KEY="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"
os.environ["GROQ_API_KEY"]= GROQ_API_KEY
groq_client =Groq(api_key=GROQ_API_KEY)
print("Groq API client initialized.")
print("Note:If you see an authentication error later,double-check your API key.")




Groq API client initialized.
Note:If you see an authentication error later,double-check your API key.


In [5]:
df = pd.read_csv('college_notes.csv')
print("Shape of dataset:", df.shape)
print("\nColumn names:", df.columns.tolist())
print("\nFiesrt 3 rows:")
print(df.head(3))


Shape of dataset: (15, 4)

Column names: ['note_id', 'subject', 'topic', 'content']

Fiesrt 3 rows:
  note_id           subject          topic  \
0    N001  Data Engineering  ETL Pipelines   
1    N002  Data Engineering  SQL Databases   
2    N003  Data Engineering  Data Cleaning   

                                             content  
0  ETL stands for Extract Transform Load. It is t...  
1  A database is an organized collection of data ...  
2  Data cleaning involves fixing or removing inco...  


In [13]:
print("Subjects in the dataset:")
print(df['subject'].value_counts())
print("\nSample of topics")
print(df[['note_id', 'subject', 'topic']].to_string(index=False))
print("\nLength of content (number of characters) for each note:")
df['content_length'] = df['content'].apply(len)
print(df[['topic', 'content_length']].head(3).to_string(index=False))

Subjects in the dataset:
subject
Data Engineering      5
Machine Learning      5
Generative AI         3
Python Programming    2
Name: count, dtype: int64

Sample of topics
note_id            subject                          topic
   N001   Data Engineering                  ETL Pipelines
   N002   Data Engineering                  SQL Databases
   N003   Data Engineering                  Data Cleaning
   N004   Data Engineering       APIs and Data Collection
   N005   Data Engineering           Big Data and PySpark
   N006   Machine Learning            Supervised Learning
   N007   Machine Learning               Model Evaluation
   N008   Machine Learning            Feature Engineering
   N009   Machine Learning                 Decision Trees
   N010   Machine Learning                  Random Forest
   N011      Generative AI          Large Language Models
   N012      Generative AI             Prompt Engineering
   N013      Generative AI Retrieval Augmented Generation
   N014 Python 

In [15]:
documents = df['content'].tolist()
ids = [f"note_{row['note_id']}" for row in df.to_dict('records')]
metadatas = [
    {"subject": row['subject'], "topic": row['topic']}
    for row in df.to_dict('records')
]

print(f"Total chunks prepared: {len(documents)}")
print(f"First documents ID: {ids[0]}")
print(f"First documents metadata: {metadatas[0]}")
print(f"First 100 chars of doc: {documents[0][:100]}...")

Total chunks prepared: 15
First documents ID: note_N001
First documents metadata: {'subject': 'Data Engineering', 'topic': 'ETL Pipelines'}
First 100 chars of doc: ETL stands for Extract Transform Load. It is the process of collecting raw data from different sourc...


In [19]:
print("Loading embedding model...")
print("(This may take 30-60 seconds on first run - model is being downloaded)")
print("(Subsequent runs will be faster as the model is cached)")

embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

print("\nEmbedding model loaded successfully.")

test_embedding = embedding_model.encode("This is test sentence.")

print(f"Test embedding shape: {test_embedding.shape}")

print(f"First 5 values of test embedding: {test_embedding[:5]}")

Loading embedding model...
(This may take 30-60 seconds on first run - model is being downloaded)
(Subsequent runs will be faster as the model is cached)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Embedding model loaded successfully.
Test embedding shape: (384,)
First 5 values of test embedding: [ 0.09194463  0.06616743 -0.00881809  0.10051527  0.01936706]


In [21]:
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection(name="college_notes_rag")

print("ChromaDB client create.")
print(f"Collection name: college_notes_rag")
print(f"Documents in collection so far: {collection.count()}")

ChromaDB client create.
Collection name: college_notes_rag
Documents in collection so far: 0


In [23]:
print("Generating embeddings for all 15 notes...")
print("This may take 15-30 seconds...")

embedding = embedding_model.encode(documents, show_progress_bar=True)
print(f"\nEmbedding matrix shape: {embedding.shape}")
embeddings_list = embedding.tolist()

collection.add(
    documents=documents,
    embeddings=embeddings_list,
    ids=ids,
    metadatas=metadatas
)
print(f"\nDocument successfully added to ChromaDB")
print(f"Total documents in collection: {collection.count()}")

Generating embeddings for all 15 notes...
This may take 15-30 seconds...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Embedding matrix shape: (15, 384)

Document successfully added to ChromaDB
Total documents in collection: 15


In [2]:
def retrieve_relevant_chunks(question,top_k=3):
    question_embedding=embedding_model.encode(question).tolist()
    results=collection.query(
        query_embeddings=question_embedding,
        n_results=top_k,
        include=['documents','metadatas']
    )
    return results

In [7]:
from sentence_transformers import SentenceTransformer
import chromadb

embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection(name="college_notes_rag")

test_question="What is ETL and how does it work in data engineering"
print(f'Test Quesstion:{test_question}')
results=retrieve_relevant_chunks(test_question,top_k=3)
print('\n Top 3 Retrieved Chunks ')
print("="*60)
for i,(doc,meta)in enumerate(zip(
    results['documents'][0],
    results['metadatas'][0]
)):

  print(f'\nResult {i+1}:')
  print(f'Subject:{meta["subject"]}')
  print(f'Topic:{meta["topic"]}')
  print(f'Content:{doc[:120]}...')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Test Quesstion:What is ETL and how does it work in data engineering

 Top 3 Retrieved Chunks 


In [4]:
def generate_rag_answer(question, context):
  """
  Send the retrieved context and questions

  Parametrs:
      question (str): The user's question
      context (str): The retrieved context chunks (formatted string)


  Returns:
     answer (str) : The LLM's generated answer
  """

  system_prompt = """You are a helpful academic assistant for engineering students.

 You will be given context retrieved from a college knowledge base, and a student's question.

 RULES:
 1.Answer ONLY using the information provided in the context below.
 2.If the answer is not found in the context, say exactly:
 "I don't have enough information in my knowledge base to answer this question"
 3.Do not use your general training knowledge.
 4.Keep answers clear, accurate, and beginner-friendly.
 5.Mention which source the information came from which possible."""

  user_prompt = f"""Context: {context}

 Student's Question: {question}"""

  response = groq_client.chat.completions.create(
      model = "llama-3.1-8b-instant",
      messages = [
          {"role": "system", "content": system_prompt},
          {"role": "user", "content": user_prompt}
      ],
      temperature = 0.1,
      max_tokens = 500
  )
  answer = response.choices[0].message.content
  return answer
print("RAG generation fucntion defined.")

RAG generation fucntion defined.


In [5]:
def generate_rag_answer(question, context):
    system_prompt = """
You are a helpful academic assistant for engineering students.

You will be given context retrieved from a college knowledge base and a student's question.

RULES:
1. Answer only using the information provided in the context.
2. If the answer is not found in the context, say exactly:
   "I don't have enough information in my knowledge base to answer this question."
3. Do not use your general training knowledge.
4. Keep answers clear, accurate, and beginner-friendly.
5. Mention which source the answer came from.
"""

    user_prompt = f"""
Context:
{context}

Question:
{question}
"""

    response = client.chat.completions.create(
        model="llama-3.1-db-instant",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.1,
        max_tokens=400
    )

    return response.choices[0].message.content
print("done")

done


In [6]:
def ask_college_assistant(question, top_k=3, verbose=True):
    """
    Complete RAG pipeline: Given a question, retrieve relevant context and generate an answer

    Parameters:
        question (str): The user's question
        top_k (int): The number of context chunks to retrieve
        verbose (bool): If True, print

    Returns:
        answer (str): The LLM's generated answer
    """

    if verbose:
        print(f'Question: {question}')
        print("=" * 60)
        print("Step 1: Retrieving relevant documents...")